# 03. Conformal shared mesh

This notebook compares the initial triangle net with the conformal shared mesh. Inserted nodes are coloured by the pinch-out boundary of the corresponding geological unit.

**Input:** `demo_model.npz`, generated by `run_model.py`. The input directory is selected by `HG3D_MODEL_DIR` (default: `outputs/model/`).

**Output:** `Local mesh refinement.png`. Figures are displayed and saved to `HG3D_FIGURE_DIR` (default: `outputs/figures/`).

**Interpretation:** The geometry is read from the model package. This notebook does not rebuild or refine the mesh.


## 1. Imports and configuration


In [ ]:
import os
import sys
from pathlib import Path

# Project paths.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "run_model.py").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start this notebook from the HarmonicGeo3D directory or a subdirectory."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np

# Input data.
MODEL_DATA_DIR = Path(
    os.environ.get("HG3D_MODEL_DIR", PROJECT_ROOT / "outputs" / "model")
)

from scripts.mesh_topology import build_mesh_topology

CM_TO_INCH = 1 / 2.54
FONT_FAMILY = ["Arial", "DejaVu Sans"]
# Unit labels and colours follow body_names from bottom to top.
BODY_LABELS = ["Coal", "Lacustrine", "Deltaic", "Fluvial", "Topsoil"]
BODY_COLORS = ["#41464B", "#849EAF", "#88AAA3", "#C7AE80", "#B8B0A3"]

FONT_SIZES = {
    "title": 7.0,
    "label": 6.0,
    "tick": 4,
    "contour": 4,
    "legend": 5.0,
    "tag": 7,
}

SELECTED_BODY_NAMES = ("body-01", "body-02", "body-03")
FIGURE_WIDTH_CM, FIGURE_HEIGHT_CM = 11.0, 7.0
PLOT_RC = {
    "font.family": FONT_FAMILY,
    "axes.unicode_minus": False,
}

# Figure output.
FIGURE_DIR = Path(
    os.environ.get("HG3D_FIGURE_DIR", PROJECT_ROOT / "outputs" / "figures")
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data loading and node classification


In [ ]:
with np.load(MODEL_DATA_DIR / "demo_model.npz", allow_pickle=False) as model_data:
    initial_nodes_xy = model_data["initial_nodes_xy"]
    initial_triangles = model_data["initial_triangles"]
    final_nodes_xy = model_data["final_nodes_xy"]
    final_triangles = model_data["final_triangles"]
    body_names = model_data["body_names"].astype(str)
    original_node_count = int(model_data["original_node_count"])
    pinch_edge_mask = model_data["pinch_edge_mask"]
topology = build_mesh_topology(final_nodes_xy, final_triangles)

if len(body_names) != len(BODY_LABELS):
    raise ValueError("The geological-unit count must match the legend configuration.")
if initial_nodes_xy.shape[0] != original_node_count:
    raise ValueError("The initial-node count must match original_node_count.")

# Inserted nodes are classified by the boundaries of the selected units.
body_name_to_index = {name: index for index, name in enumerate(body_names)}
missing_bodies = [
    name for name in SELECTED_BODY_NAMES if name not in body_name_to_index
]
if missing_bodies:
    raise ValueError(
        f"The model is missing the selected geological units: {missing_bodies}"
    )
selected_body_indices = [body_name_to_index[name] for name in SELECTED_BODY_NAMES]

inserted_node_ids = np.arange(original_node_count, len(final_nodes_xy))
inserted_node_ids_by_body = {}
for body_index in selected_body_indices:
    body_edge_nodes = np.unique(topology.edges[pinch_edge_mask[:, body_index]])
    inserted_node_ids_by_body[body_index] = inserted_node_ids[
        np.isin(inserted_node_ids, body_edge_nodes)
    ]
classified_node_ids = np.concatenate(list(inserted_node_ids_by_body.values()))
if not np.array_equal(np.sort(classified_node_ids), inserted_node_ids):
    raise ValueError(
        "Each inserted node must belong to exactly one selected geological unit."
    )

## 3. Plot the initial and conformal meshes


In [ ]:
with plt.rc_context(PLOT_RC):
    fig, (initial_ax, final_ax) = plt.subplots(
        1,
        2,
        figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, FIGURE_HEIGHT_CM * CM_TO_INCH),
        dpi=600,
        layout="constrained",
        sharey=True,
    )

    # Both panels use the same inserted-node colours and drawing order.
    for mesh_ax, mesh_xy, mesh_triangles, borehole_xy in (
        (initial_ax, initial_nodes_xy, initial_triangles, initial_nodes_xy),
        (
            final_ax,
            final_nodes_xy,
            final_triangles,
            final_nodes_xy[:original_node_count],
        ),
    ):
        mesh_ax.triplot(
            mesh_xy[:, 0],
            mesh_xy[:, 1],
            mesh_triangles,
            color="0.65",
            linewidth=0.45,
            zorder=1,
        )
        mesh_ax.scatter(
            borehole_xy[:, 0],
            borehole_xy[:, 1],
            s=6,
            facecolors="white",
            edgecolors="#333333",
            linewidths=0.5,
            label="Existing boreholes",
            zorder=2,
        )
        for body_index in selected_body_indices:
            node_ids = inserted_node_ids_by_body[body_index]
            mesh_ax.scatter(
                final_nodes_xy[node_ids, 0],
                final_nodes_xy[node_ids, 1],
                s=10,
                facecolors=BODY_COLORS[body_index],
                edgecolors="white",
                linewidths=0.5,
                label=f"{BODY_LABELS[body_index]} nodes",
                zorder=3,
            )
    for body_index in selected_body_indices:
        pinch_edges = topology.edges[pinch_edge_mask[:, body_index]]
        for edge_index, edge in enumerate(pinch_edges):
            final_ax.plot(
                final_nodes_xy[edge, 0],
                final_nodes_xy[edge, 1],
                color=BODY_COLORS[body_index],
                linewidth=0.8,
                label=(
                    f"{BODY_LABELS[body_index]} pinch-out" if edge_index == 0 else None
                ),
                zorder=3,
            )

    # Equal XY scales preserve the geometry of the map.
    x_padding = np.ptp(final_nodes_xy[:, 0]) * 0.03
    y_padding = np.ptp(final_nodes_xy[:, 1]) * 0.03
    x_limits = (
        final_nodes_xy[:, 0].min() - x_padding,
        final_nodes_xy[:, 0].max() + x_padding,
    )
    y_limits = (
        final_nodes_xy[:, 1].min() - y_padding,
        final_nodes_xy[:, 1].max() + y_padding,
    )
    for panel_ax, panel_label, is_initial in (
        (final_ax, "(b)", False),
        (initial_ax, "(a)", True),
    ):
        panel_ax.set_xlim(*x_limits)
        panel_ax.set_ylim(*y_limits)
        panel_ax.set_aspect("equal", adjustable="box")
        if is_initial:
            panel_ax.set_box_aspect(
                (y_limits[1] - y_limits[0]) / (x_limits[1] - x_limits[0])
            )

        for spine in panel_ax.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#333333")
        panel_ax.set_xlabel("X (m)", family=FONT_FAMILY, fontsize=6)
        if is_initial:
            panel_ax.set_ylabel("Y (m)", family=FONT_FAMILY, fontsize=6)
        panel_ax.set_axisbelow(True)

        panel_ax.grid(axis="both", alpha=0.8, color="lightgray", linewidth=0.25)

        panel_ax.tick_params(
            axis="x",
            which="both",
            top=True,
            bottom=True,
            labeltop=False,
            labelbottom=True,
            direction="in",
            labelsize=FONT_SIZES["tick"],
            length=2,
            width=0.5,
        )
        panel_ax.tick_params(
            axis="y",
            which="both",
            left=True,
            right=True,
            labelright=False,
            labelleft=is_initial,
            direction="in",
            length=2,
            width=0.5,
            rotation=90,
            **({"labelsize": FONT_SIZES["tick"]} if is_initial else {}),
        )
        panel_ax.ticklabel_format(style="plain", useOffset=False)
        plt.setp(panel_ax.get_xticklabels(), family=FONT_FAMILY, ha="center", va="top")
        if is_initial:
            plt.setp(
                panel_ax.get_yticklabels(),
                family=FONT_FAMILY,
                rotation=90,
                ha="center",
                va="center",
            )

        panel_ax.text(
            0.02,
            0.98,
            panel_label,
            transform=panel_ax.transAxes,
            ha="left",
            va="top",
            family=FONT_FAMILY,
            fontsize=7,
        )

    legend = final_ax.legend(
        loc="upper right",
        borderaxespad=0.2,
        frameon=True,
        framealpha=0.9,
        edgecolor="0.5",
        fontsize=4,
        handlelength=1.8,
    )
    legend.get_frame().set_linewidth(0.4)
    plt.setp(legend.get_texts(), family=FONT_FAMILY)
    plt.savefig(FIGURE_DIR / "Local mesh refinement.png", dpi=600)
    plt.show()